<a href="https://colab.research.google.com/github/julmiha25-sys/MathStatistica/blob/main/%D0%90-%D0%92_%D1%82%D0%B5%D1%81%D1%82/%D0%90%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_%D0%90_%D0%92_%D1%82%D0%B5%D1%81%D1%82%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
from scipy import stats
df= pd.read_csv("AB_Test_revenue_only.csv", sep=";")
#Сколько уникальных пользователей участвовало в эксперименте всего?
uniq=df['user_id'].nunique()
print("Количество уникальных пользователей:", uniq)
#Какие названия были использованы для экспериментальной и контрольной групп?
print("Названия групп:", df['variant_name'].unique())
#Есть ли в дата сете пользователи, которые попали в обе группы?
res=df.groupby('user_id')['variant_name'].nunique()
double=res[res>1]
print("Пользователи, которые попали в обе группы:\n", double)
#Сколько таких?
count_double=len(double)
double=double.reset_index()
print("Количество пользователей, которые попали в обе группы:", count_double)
#Анатолично подсчитаем через множество set
set_A=set(df[df['variant_name']=='control']['user_id'])
set_B=set(df[df['variant_name']=='variant']['user_id'])
set_double=set_A.intersection(set_B)
print("Количество пользователей, которые попали в обе группы (считаем через пересечение множеств):", len(set_double))
#Какой процент пользователей вы удалили из-за ошибки системы сплитования? Округлите ответ до сотых.
print("% пользователей удалили из-за ошибки системы сплитования:", round(count_double/uniq*100,2),"%")
#На сколько процентов отличается количество наблюдений в экспериментальных группах? (округлите до сотых процента)
df_new=df[~df['user_id'].isin(double['user_id'])]
A=df_new[df_new['variant_name']=='control']
B=df_new[df_new['variant_name']=='variant']
print("На сколько % отличается количество наблюдений в экспериментальных группах:", round((len(B)-len(A))/len(A)*100,2),"%")
#Отличие < 1% - сплитование допустимо
#Отличаются ли стат значимо группы в А/В-тесте?
stat=stats.ttest_ind(A['revenue'], B['revenue'])
print(f"Соотношение выручки для групп А/B - статистика: {stat}")
if stat.pvalue <= 0.05:
   print("p_value <= 0.05 - гипотезу о равенстве распределений выручки в группах A и B отвергаем на уровне значимости а=0.05")
else:
   print("p_value > 0.05 - гипотезу о равенстве распределений выручки в группах A и B не отвергаем на уровне значимости а=0.05")
#p_value=0 < 0.05 - гопотезу о равенстве распределений выручки в группах A1 и A2 отвергаем на уровне значимости а=0.05
#Какая группа показала лучшие результаты?
rev_A=round(A['revenue'].sum(),2)
rev_B=round(B['revenue'].sum(),2)
if rev_A>rev_B:
  print("Выручка группы А больше B:",rev_A, rev_B)
elif  rev_B>rev_A:
  print("Выручка группы B больше A:",rev_B, rev_A)
else:
  print("Выручки в группах А и B одинаковы:",rev_B)
#Вывод:группа В показала лучшие результаты

# Проверка выбросов выручки (просто диагностика)
def remove_outliers_iqr(data):
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    cleaned_data = data[(data > lower_bound) & (data < upper_bound)]
    delete_data = data[(data <= lower_bound) | (data >= upper_bound)]
    return cleaned_data, delete_data


cleaned_data_A, delete_data_A = remove_outliers_iqr(A['revenue'])
cleaned_data_B, delete_data_B = remove_outliers_iqr(B['revenue'])
print("Выбросы в группе А:", delete_data_A)
print("Выбросы в группе А:", delete_data_B)

Количество уникальных пользователей: 6324
Названия групп: ['control' 'variant']
Пользователи, которые попали в обе группы:
 user_id
3        2
10       2
18       2
25       2
40       2
        ..
9978     2
9979     2
9982     2
9996     2
10000    2
Name: variant_name, Length: 1541, dtype: int64
Количество пользователей, которые попали в обе группы: 1541
Количество пользователей, которые попали в обе группы (считаем через пересечение множеств): 1541
% пользователей удалили из-за ошибки системы сплитования: 24.37 %
На сколько % отличается количество наблюдений в экспериментальных группах: 0.59 %
Соотношение выручки для групп А/B - статистика: TtestResult(statistic=np.float64(-20.06371419335301), pvalue=np.float64(9.473805630708502e-87), df=np.float64(6068.0))
p_value <= 0.05 - гипотезу о равенстве распределений выручки в группах A и B отвергаем на уровне значимости а=0.05
Выручка группы B больше A: 18224.07 15052.3
Выбросы в группе А: 1054    11.341950
2060    10.858192
3118    12.60